In [1]:
"""
Planner agent
Research agent
paper agent 
verificqtion agent
summarization agent
report agent
"""

'\nPlanner agent\nResearch agent\npaper agent \nverificqtion agent\nsummarization agent\nreport agent\n'

In [5]:
import arxiv
import langchain_community
print("arxiv:", arxiv.__version__)
print("langchain_community:", langchain_community.__version__)

arxiv: 4.0.1
langchain_community: 0.4.2


/var/folders/ps/44m5_0n97g99m3c7tf4h0qmc0000gn/T/ipykernel_34291/2665778139.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [6]:
import logging
import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("research_assistant.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

In [8]:
import openai
print(openai.__version__)

2.53.0


In [10]:
import openai
import langchain
import langchain_openai

print(openai.__version__)
print(langchain.__version__)

2.53.0
1.3.14


In [11]:
import langchain
import langchain_core
import langchain_community
import langchain_openai

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-community:", langchain_community.__version__)
print("langchain-openai:", langchain_openai.__version__)

langchain: 1.3.14
langchain-core: 1.5.3
langchain-community: 0.4.2
langchain-openai: 1.4.2


In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
print(f"LLM ready: {llm.model_name}")
llm.invoke("welcome me")

LLM ready: gpt-4o


2026-08-07 16:18:17,297 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


AIMessage(content="Welcome! I'm glad you're here. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 9, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_541568ae70', 'id': 'chatcmpl-EAMFEmgurUPfqsFcmIW2B0aYJj2iQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fde17-4600-7f11-bf4c-5da68cee2071-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 14, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
"""usage = response.usage_metadata

logger.info(f"Input Tokens : {usage['input_tokens']}")
logger.info(f"Output Tokens : {usage['output_tokens']}")
logger.info(f"Total Tokens : {usage['total_tokens']}")"""

'usage = response.usage_metadata\n\nlogger.info(f"Input Tokens : {usage[\'input_tokens\']}")\nlogger.info(f"Output Tokens : {usage[\'output_tokens\']}")\nlogger.info(f"Total Tokens : {usage[\'total_tokens\']}")'

In [14]:
from typing import Annotated,TypedDict,List
class State(TypedDict):
    question: str
    plan:str
    research_task: str
    verification_task: str
    research: list
    paper: list
    summary: str
    report: str
    use_research: bool
    use_paper: bool

In [15]:
from pydantic import BaseModel
class strc(BaseModel):
    quries: List[str]

In [16]:
from pydantic import BaseModel, Field
from typing import List
import json

from langchain_core.messages import SystemMessage, HumanMessage


class PlannerOutput(BaseModel):

    plan: str = Field(
        description="Detailed research plan."
    )

    use_research: bool = Field(
        description="Whether the web Research Agent should be used."
    )

    use_paper: bool = Field(
        description="Whether the academic Paper Agent should be used."
    )


def planner_agent(state: State):

    logger.info("planner_agent just started")

    messages = [

        SystemMessage(content="""
You are an AI Research Planning Agent.

Your responsibility is to analyze the user's research question
and create a clear research plan.

Break the question into smaller research tasks that can be
completed by specialized agents.

Available agents:

1. Research Agent
   - searches technical blogs, documentation, and web sources.

2. Paper Agent
   - finds and analyzes academic research papers.

3. Verification Agent
   - checks information accuracy across sources.

4. Summarization Agent
   - combines findings into structured notes.

5. Report Agent
   - generates the final research report.

Decide whether web research and academic paper research
are needed.

Return ONLY valid JSON in exactly this format:

{
    "plan": "Detailed step-by-step research plan",
    "use_research": true,
    "use_paper": false
}

Rules:

- use_research should be true if web sources,
  documentation, blogs, or current technical information
  are useful.

- use_paper should be true if academic research papers,
  scientific evidence, algorithms, or research findings
  are useful.

- Both can be true.

- Do not answer the user's question.
- Only create the research workflow plan.
        """),

        HumanMessage(
            content=state["question"]
        )
    ]

    response = llm.invoke(messages)

    print("MODEL RESPONSE:")
    print(response.content)

    logger.info("planner_agent completed")

    # Get the text returned by the LLM
    content = response.content.strip()

    # Remove markdown code fences if the model adds them
    if content.startswith("```json"):
        content = content[7:]

    elif content.startswith("```"):
        content = content[3:]

    if content.endswith("```"):
        content = content[:-3]

    content = content.strip()

    # Convert JSON text into Python dictionary
    data = json.loads(content)

    # Validate the dictionary using Pydantic
    planner_output = PlannerOutput(**data)

    return {
        "plan": planner_output.plan,
        "use_research": planner_output.use_research,
        "use_paper": planner_output.use_paper
    }

In [17]:
from pydantic import BaseModel
from typing import List


class SearchQueries(BaseModel):

    queries: List[str]

In [18]:
state = {
    "question": "Explain the latest advances in RAG systems",
    "plan": "",
    "use_research": False,
    "use_paper": False
}
state['plan']=planner_agent(state)['plan']

2026-08-07 16:18:17,379 | INFO | planner_agent just started
2026-08-07 16:18:21,192 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:18:21,199 | INFO | planner_agent completed


MODEL RESPONSE:
{
    "plan": "1. Use the Research Agent to search for the latest advancements in Retrieval-Augmented Generation (RAG) systems by exploring technical blogs, industry reports, and documentation from AI research labs and companies. 2. Use the Paper Agent to find recent academic papers that discuss new algorithms, models, or techniques in RAG systems. 3. Use the Verification Agent to cross-check the information gathered from web sources and academic papers to ensure accuracy and consistency. 4. Use the Summarization Agent to compile the verified information into structured notes, highlighting key advancements, methodologies, and applications. 5. Use the Report Agent to generate a comprehensive research report detailing the latest advances in RAG systems, including potential implications and future directions.",
    "use_research": true,
    "use_paper": true
}


In [19]:
tools=TavilySearch(max_results=3)

def Research_agent(state: State):
    logger.info("Research_agent just started")
    messages=[

        SystemMessage(content="""
            You are an AI Research Agent.

            Your responsibility is to gather information from various sources based on the research plan provided by the Planner Agent.

            Use the TavilySearch tool to find relevant information from technical blogs, documentation, and web sources.

            Collect important concepts, industry updates, and any other relevant information.

            Do not answer the user's question directly.
            Only gather information as per the research plan.

            Example:

            Research Plan:
            "Find recent blogs and technical articles about RAG techniques."

            Output:
            - Collected information from various sources.
            - Key concepts and industry updates related to RAG techniques.
            """
        ),
        HumanMessage(content=f"{state['question']} and you can refer to the plan {state['plan']} for guidance on what to research. Please provide a summary of your findings.")
    ]
    print(state['question'])
    
    response=llm.with_structured_output(strc,include_raw=True).invoke(messages)
    results = []

    for query in response['parsed'].quries:

        try:

            search_results = tools.run(
                query
            )

            results.append({
                "query": query,
                "results": search_results
            })

        except Exception as e:

            logger.exception(
                f"Search failed for query: {query}"
            )

            results.append({
                "query": query,
                "error": str(e)
            })

    logger.info(
        f"Research_agent completed with "
        f"{len(results)} results"
    )

    return {
        "research": results
    }

    

In [20]:
state = {"question": "what is langchain"}
state["plan"] = planner_agent(state)["plan"]
Research_agent(state)

2026-08-07 16:18:21,221 | INFO | planner_agent just started
2026-08-07 16:18:23,481 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:18:23,485 | INFO | planner_agent completed
2026-08-07 16:18:23,488 | INFO | Research_agent just started


MODEL RESPONSE:
{
    "plan": "1. Use the Research Agent to gather information from technical blogs, documentation, and web sources about LangChain, including its purpose, features, and applications. 2. Use the Verification Agent to ensure the accuracy of the information collected from various sources. 3. Use the Summarization Agent to compile the verified information into structured notes. 4. Use the Report Agent to generate a comprehensive research report on LangChain.",
    "use_research": true,
    "use_paper": false
}
what is langchain


2026-08-07 16:18:25,492 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:18:44,849 | INFO | Research_agent completed with 5 results


{'research': [{'query': 'LangChain purpose and features',
   'results': "No search results found for 'LangChain purpose and features'. Suggestions: Try a general search using 'general' topic. Try modifying your search parameters with one of these approaches."},
  {'query': 'LangChain applications',
   'results': {'query': 'LangChain applications',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [{'url': 'https://aws.amazon.com/what-is/langchain',
      'title': 'What is LangChain?',
      'content': 'LangChain is an open source framework for building applications based on large language models (LLMs). LLMs are large deep-learning models pre-trained on large amounts of data that can generate responses to user queries—for example, answering questions or creating images from text-based prompts. LangChain provides tools and abstractions to improve the customization, accuracy, and relevancy of the information the models generate. For example, developers

In [21]:
state['research']=Research_agent(state)["research"]



2026-08-07 16:18:44,872 | INFO | Research_agent just started


what is langchain


2026-08-07 16:18:46,076 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:18:49,471 | INFO | Research_agent completed with 5 results


In [22]:
import arxiv

arxiv_client = arxiv.Client(page_size=5, delay_seconds=5.0, num_retries=3)

def search_arxiv(query, max_results=3):
    search = arxiv.Search(query=query, max_results=max_results)
    docs = []
    for r in arxiv_client.results(search):
        docs.append(
            f"Published: {r.published.date()}\n"
            f"Title: {r.title}\n"
            f"Authors: {', '.join(a.name for a in r.authors)}\n"
            f"Summary: {r.summary[:2000]}"
        )
    return "\n\n".join(docs) if docs else "No good Arxiv Result was found"

In [23]:


def Paper_agent(state: State):
    logger.info("Paper_agent just started")
    messages=[

        SystemMessage(content="""
            You are an AI Paper Agent.

            Your responsibility is to find and analyze academic research papers based on the research plan provided by the Planner Agent.

            Use the Arxiv tool to find relevant academic papers.

            Extract key contributions, methodologies, and findings from the papers.

            Do not answer the user's question directly.
            Only gather information as per the research plan.

            Example:

            Research Plan:
            "Find recent research papers about RAG improvements."

            Output:
            - Collected information from various academic papers.
            - Key contributions and findings related to RAG techniques.
            """
        ),
        HumanMessage(content=f"{state['question']} and you can refer to the plan {state['plan']} for guidance on what to research. Please provide a summary of your findings.")
    ]
    
    output=llm.with_structured_output(strc,include_raw=True).invoke(messages)
    print(output["raw"].usage_metadata)
    
    op=[]

    for query in output['parsed'].quries[:3]:
        try:

            sr = search_arxiv(
                query
            )

            op.append({
                "query": query,
                "results": sr
            })

        except Exception as e:

            logger.exception(
                f"arXiv search failed: {query}"
            )

            op.append({
                "query": query,
                "error": str(e)
            })

    logger.info(
        f"Paper_agent completed with "
        f"{len(op)} results"
    )
    return{
        "paper":op
        
    }
    

In [24]:
state['paper']=Paper_agent(state)['paper']

2026-08-07 16:18:49,500 | INFO | Paper_agent just started
2026-08-07 16:18:51,606 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:18:51,621 | INFO | Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=Find+recent+research+papers+about+LangChain%27s+purpose+and+applications.&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=5


{'input_tokens': 286, 'output_tokens': 58, 'total_tokens': 344, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


2026-08-07 16:18:51,857 | INFO | Got first page: 5 of 1773732 total results
2026-08-07 16:18:51,858 | INFO | Sleeping: 4.992475 seconds
2026-08-07 16:18:56,855 | INFO | Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=Look+for+academic+papers+discussing+the+features+of+LangChain.&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=5
2026-08-07 16:18:57,752 | INFO | Got first page: 5 of 1182561 total results
2026-08-07 16:18:57,752 | INFO | Sleeping: 4.997673 seconds
2026-08-07 16:19:02,757 | INFO | Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=Search+for+studies+or+papers+that+explore+the+technical+aspects+of+LangChain.&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=5
2026-08-07 16:19:03,593 | INFO | Got first page: 5 of 1623045 total results
2026-08-07 16:19:03,594 | INFO | Paper_agent completed with 3 results


In [25]:
def Verfication_agent(state: State):
    logger.info("Verfication_agent just started")
    messages2=[

        SystemMessage(content="""
            You are an AI Verification Agent.

            Your responsibility is to check the accuracy of the information gathered by the Research and Paper Agents.

            Compare findings from multiple sources and identify any conflicting information.

            Ensure that the information is reliable and accurate.

            Do not answer the user's question directly.
            Only verify the information as per the research plan.

            Example:

            Research Plan:
            "Verify the accuracy of information about RAG techniques."

            Output:
            - Verified information from various sources.
            - Identified any conflicting information.
            """
        ),
        HumanMessage(content=f"{state['question']} and you can refer to the plan {state['plan']} for guidance on what to verify. Please provide a summary of your findings. and you can refer to the research findings {state.get('research', [])} and paper findings {state.get('paper', [])} for comparison.verify all the information and also provide the mistakes if any and also provide the correct information if any mistakes are found.")
    ]
    respons=llm.invoke(messages2)
    print(respons.usage_metadata)
    

    
    print(respons.content)
    logger.info("Verfication_agent completed")

    return{
        "verification_task":respons.content
    }

In [26]:
Verfication_agent(state)

2026-08-07 16:19:03,620 | INFO | Verfication_agent just started
2026-08-07 16:19:10,378 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:10,386 | INFO | Verfication_agent completed


{'input_tokens': 7944, 'output_tokens': 495, 'total_tokens': 8439, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Research Plan:
1. Verify the accuracy of information about LangChain, including its purpose, features, and applications.

Output:
- Verified information from various sources.
- Identified any conflicting information.

### Verified Information:

1. **Purpose and Features of LangChain:**
   - LangChain is an open-source orchestration framework designed to simplify the development of applications using large language models (LLMs). It provides tools and components to connect LLMs with various data sources, enabling the creation of complex, multi-step workflows. This information is consistently supported by multiple sources, including technical blogs and official documentation.
   - Key features include prompt engineering, data connection and retrieval, and the ability to integrate various models for enhanced language

{'verification_task': "Research Plan:\n1. Verify the accuracy of information about LangChain, including its purpose, features, and applications.\n\nOutput:\n- Verified information from various sources.\n- Identified any conflicting information.\n\n### Verified Information:\n\n1. **Purpose and Features of LangChain:**\n   - LangChain is an open-source orchestration framework designed to simplify the development of applications using large language models (LLMs). It provides tools and components to connect LLMs with various data sources, enabling the creation of complex, multi-step workflows. This information is consistently supported by multiple sources, including technical blogs and official documentation.\n   - Key features include prompt engineering, data connection and retrieval, and the ability to integrate various models for enhanced language understanding and generation. LangChain's modular structure allows developers to create tailored solutions for diverse applications, as conf

In [27]:
state['verification_task']=Verfication_agent(state)['verification_task']

2026-08-07 16:19:10,401 | INFO | Verfication_agent just started
2026-08-07 16:19:16,183 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:16,188 | INFO | Verfication_agent completed


{'input_tokens': 7944, 'output_tokens': 428, 'total_tokens': 8372, 'input_token_details': {'audio': 0, 'cache_read': 7808}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Research Plan: Verify the accuracy of information about LangChain, including its purpose, features, and applications.

### Verified Information:

1. **Purpose and Features of LangChain:**
   - LangChain is an open-source orchestration framework designed to simplify the development of applications using large language models (LLMs). It provides tools and components to connect LLMs with various data sources, enabling the creation of complex, multi-step workflows. This information is consistently supported by multiple sources, including technical blogs and official documentation.
   - Key features include prompt engineering, data connection and retrieval, chains, agents, and memory. LangChain enhances language understanding and generation by integrating various models, allowing developers to leverage the strengths

In [28]:
def summarization_agent(state: State):
    logger.info("summarization_agent just started")
    messages3=[

        SystemMessage(content="""
            You are an AI Summarization Agent.

            Your responsibility is to combine the findings from the Research, Paper, and Verification Agents into structured notes.

            Create a clear and concise summary of the information gathered.

            Do not answer the user's question directly.
            Only summarize the information as per the research plan.

            Example:

            Research Plan:
            "Summarize the findings  about the question"

            Output:
            - Structured notes summarizing the key points from all sources.
            """
        ),
        HumanMessage(content=f"{state['question']} and you can refer to the plan {state['plan']} for guidance on what to summarize. Please provide a summary of your findings. and you can refer to the research findings {state.get('research', [])} and paper findings {state.get('paper', [])} for comparison. and moreover you can also refer to the verification findings for more information{state.get('verification_task', '')}. Please provide a structured summary of all the information gathered.")
    ]
    response=llm.invoke(messages3)
    print(response.usage_metadata)
    logger.info("summarization_agent ended")

    return{
        "summary":response.content
    }   

In [29]:
summarization_agent(state)

2026-08-07 16:19:16,211 | INFO | summarization_agent just started
2026-08-07 16:19:26,324 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:26,332 | INFO | summarization_agent ended


{'input_tokens': 8365, 'output_tokens': 491, 'total_tokens': 8856, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


{'summary': "### Structured Summary of LangChain\n\n#### Purpose and Features\n- **LangChain** is an open-source orchestration framework designed to simplify the development of applications using large language models (LLMs).\n- It provides tools and components to connect LLMs with various data sources, enabling the creation of complex, multi-step workflows.\n- Key features include:\n  - **Prompt Engineering**: Enhances the creation and refinement of prompts for LLMs to ensure accurate and relevant responses.\n  - **Data Connection and Retrieval**: Facilitates the integration of LLMs with external data sources.\n  - **Chains, Agents, and Memory**: Supports the orchestration of complex behaviors and interactions with models.\n  - **Customization and Flexibility**: Allows developers to mix and match components to create tailored solutions.\n\n#### Applications\n- LangChain is used to build a variety of applications powered by language models, such as:\n  - **Chatbots and Conversational A

In [30]:
state['summary']=summarization_agent(state)['summary']

2026-08-07 16:19:26,345 | INFO | summarization_agent just started
2026-08-07 16:19:30,790 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:30,799 | INFO | summarization_agent ended


{'input_tokens': 8365, 'output_tokens': 503, 'total_tokens': 8868, 'input_token_details': {'audio': 0, 'cache_read': 8192}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [31]:
def Report_agent(state: State):
    logger.info("generating final report")
    messages = [
        SystemMessage(content="""
            You are an AI Report Generation Agent.

            Your responsibility is to write a final, professional research report
            based on the structured summary notes provided.

            Structure the report with these sections:
            1. Executive Summary — a short, high-level overview (3-5 sentences).
            2. Key Findings — the main concepts and developments, organized clearly.
            3. Academic Research — contributions and findings from academic papers.
            4. Technology / Industry Comparison — if applicable, compare approaches or tools discussed.
            5. Points of Uncertainty or Disagreement — where sources conflicted or coverage was thin.
            6. Conclusion — a brief closing synthesis.

            Write in clear, professional prose suitable for sharing with a
            technical audience. Do not introduce facts that are not present in
            the provided summary notes.
        """),
        HumanMessage(content=f"""
            Original question: {state['question']}

            Summary notes:
            {state['summary']}

            Please write the final report.
            before giving the final report please check all info like {state.get('research', [])} and {state.get('paper', [])} and {state.get('verification_task', '')} and {state['summary']} and give the final report.
            and also include the references of the papers and blogs used in the report.also provide the research papers and blogs used in the report in the references section of the report.
        """)
    ]

    response = llm.invoke(messages)
    print(response.usage_metadata)
    print(response.content)
    logger.info("final report completed")

    return {
        "report": response.content
    }

In [32]:
def route_after_planner(state: State):

    use_research = state.get(
        "use_research",
        True
    )
    use_paper = state.get(
        "use_paper",
        True
    )
    if use_research and use_paper:
        return "both"
    if use_research:
        return "research"
    if use_paper:
        return "paper"
    
    return "research"

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(State)


def route_after_planner(state: State):
    use_research = state.get("use_research", True)
    use_paper = state.get("use_paper", True)
    if use_research and use_paper:
        return "both"
    if use_research:
        return "research_only"
    if use_paper:
        return "paper_only"
    return "both"

def route_after_research(state: State):
    if state.get("use_paper", True):
        return "paper"
    return "verification"

graph = StateGraph(State)

graph.add_node("planner", planner_agent)
graph.add_node("research", Research_agent)
graph.add_node("paper", Paper_agent)
graph.add_node("verification", Verfication_agent)
graph.add_node("summary", summarization_agent)
graph.add_node("report", Report_agent)

graph.set_entry_point("planner")

graph.add_conditional_edges(
    "planner",
    route_after_planner,
    {
        "both": "research",
        "research_only": "research",
        "paper_only": "paper",
    }
)

graph.add_conditional_edges(
    "research",
    route_after_research,
    {
        "paper": "paper",
        "verification": "verification"
    }
)

graph.add_edge("paper", "verification")
graph.add_edge("verification", "summary")
graph.add_edge("summary", "report")
graph.add_edge("report", END)

app = graph.compile()



import time

start = time.time()

app = graph.compile()

end = time.time()

print(f"Execution Time : {end-start:.2f} sec")

result = app.invoke({"question": "what is langchain"})
print(result['report'])

2026-08-07 16:19:30,952 | INFO | planner_agent just started


Execution Time : 0.00 sec


2026-08-07 16:19:32,942 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:32,948 | INFO | planner_agent completed
2026-08-07 16:19:32,951 | INFO | Research_agent just started


MODEL RESPONSE:
{
    "plan": "1. Use the Research Agent to gather information from technical blogs, documentation, and web sources about LangChain, including its purpose, features, and applications. 2. Use the Verification Agent to ensure the accuracy of the information collected from various sources. 3. Use the Summarization Agent to compile the verified information into structured notes. 4. Use the Report Agent to generate a comprehensive research report on LangChain.",
    "use_research": true,
    "use_paper": false
}
what is langchain


2026-08-07 16:19:33,999 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:35,177 | INFO | Research_agent completed with 5 results
2026-08-07 16:19:35,180 | INFO | Verfication_agent just started
2026-08-07 16:19:44,445 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:44,449 | INFO | Verfication_agent completed
2026-08-07 16:19:44,451 | INFO | summarization_agent just started


{'input_tokens': 4876, 'output_tokens': 525, 'total_tokens': 5401, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
### Verified Information on LangChain

1. **Purpose and Features:**
   - LangChain is a versatile framework designed for building applications powered by large language models (LLMs). It is open-source and facilitates the development of LLM-driven applications by providing tools and components to connect LLMs with various data sources.
   - Key features include prompt engineering, data connection and retrieval, and the ability to create complex, multi-step workflows. It supports customization and flexibility through its modular structure, allowing developers to mix and match components to create tailored solutions.
   - LangChain enhances language understanding and generation by integrating various models, leading to improved language processing capabilities.

2. **Applications:**
   - LangChain is used to develop

2026-08-07 16:19:49,039 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:19:49,043 | INFO | summarization_agent ended
2026-08-07 16:19:49,046 | INFO | generating final report
2026-08-07 16:19:49,142 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-07 16:19:49,145 | INFO | Retrying request to /chat/completions in 2.972000 seconds


{'input_tokens': 5394, 'output_tokens': 428, 'total_tokens': 5822, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


2026-08-07 16:20:00,420 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:20:00,427 | INFO | final report completed


{'input_tokens': 6251, 'output_tokens': 792, 'total_tokens': 7043, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
# LangChain: A Comprehensive Overview

## Executive Summary

LangChain is an open-source framework designed to streamline the development of applications powered by large language models (LLMs). It offers a suite of tools and components that facilitate the integration of LLMs with various data sources, enabling the creation of complex, multi-step workflows. Widely adopted across industries such as business services, software, and finance, LangChain is recognized for its modularity, flexibility, and rapid growth as an open-source project. The framework is supported by comprehensive documentation and a robust community, making it a valuable resource for developers working with LLMs.

## Key Findings

1. **Purpose and Features:**
   - LangChain is designed to facilitate the development of LLM-powered applications.
  

In [34]:
print(result.keys())

dict_keys(['question', 'plan', 'verification_task', 'research', 'summary', 'report', 'use_research', 'use_paper'])


In [35]:
print(graph.nodes.keys())

dict_keys(['planner', 'research', 'paper', 'verification', 'summary', 'report'])


In [36]:
print(type(app))
print("App is ready")

<class 'langgraph.graph.state.CompiledStateGraph'>
App is ready


In [40]:
import json, time
questions = [
    "What are the latest advances in Retrieval Augmented Generation?",
    "How does QLoRA differ from full fine-tuning for LLMs?",
    "What are the key challenges in multi-agent AI systems?",
    "How do vector databases compare for production RAG systems?",
    "What are the tradeoffs between RAG and fine-tuning?"
]
questions_list = []
answers_list = []
print("Running pipeline...")
for i, q in enumerate(questions):
    print(f"{i+1}/5: {q[:50]}...")
    try:
        result = app.invoke({"question": q})
        questions_list.append(q)
        answers_list.append(result.get("report", ""))
        print("Done")
    except Exception as e:
        print(f" Failed: {e}")
        questions_list.append(q)
        answers_list.append("")
    time.sleep(3)

print(f"\nCollected {len(answers_list)} reports")
print("="*50)

scores = []

print("\nEvaluating reports...")
for i, (q, a) in enumerate(zip(questions_list, answers_list)):
    print(f"\nEvaluating {i+1}/{len(questions_list)}: {q[:45]}...")
    
    try:
        response = llm.invoke(f"""Rate this research report.

Question: {q}

Report: {a[:2000]}

Return ONLY this JSON with float values between 0.0 and 1.0:
{{"faithfulness": 0.0, "answer_relevancy": 0.0}}

- faithfulness: are facts accurate and grounded?
- answer_relevancy: does it answer the question?

Start your response with {{ and end with }}""")

        content = response.content.strip()
        start = content.find("{")
        end = content.rfind("}") + 1
        score = json.loads(content[start:end])
        
        f = round(float(score.get("faithfulness", 0.0)), 2)
        r = round(float(score.get("answer_relevancy", 0.0)), 2)
        scores.append({"f": f, "r": r})
        print(f"  Faithfulness: {f} | Relevancy: {r}")
        
    except Exception as e:
        print(f"  Failed: {e}")
        print(f"  Response was: {response.content[:300]}")
        scores.append({"f": 0.0, "r": 0.0})
    
    time.sleep(2)


avg_f = round(sum(s["f"] for s in scores) / len(scores), 2)
avg_r = round(sum(s["r"] for s in scores) / len(scores), 2)

print("\n" + "="*50)
print("   MULTI-AGENT RESEARCH ASSISTANT")
print("   EVALUATION RESULTS")
print("="*50)
print(f"   Questions Evaluated : {len(scores)}")
print(f"   Faithfulness        : {avg_f}")
print(f"   Answer Relevancy    : {avg_r}")
print("="*50)
print(f"\n Resume bullet:")
print(f"   Evaluated using LLM-as-judge across {len(scores)} test")
print(f"   queries achieving faithfulness of {avg_f} and")
print(f"   answer relevancy of {avg_r}")

2026-08-07 16:47:32,879 | INFO | planner_agent just started


Running pipeline...
1/5: What are the latest advances in Retrieval Augmente...


2026-08-07 16:47:36,189 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:47:36,204 | INFO | planner_agent completed
2026-08-07 16:47:36,206 | INFO | Research_agent just started


MODEL RESPONSE:
{
    "plan": "1. Use the Research Agent to search for the latest technical blogs, documentation, and web sources discussing recent advances in Retrieval Augmented Generation (RAG). 2. Use the Paper Agent to find and analyze recent academic research papers on RAG to understand the scientific advancements and methodologies. 3. Use the Verification Agent to cross-check the information gathered from both web sources and academic papers for accuracy and consistency. 4. Use the Summarization Agent to compile the verified information into structured notes highlighting key advancements, methodologies, and applications of RAG. 5. Use the Report Agent to generate a comprehensive research report summarizing the latest advances in Retrieval Augmented Generation.",
    "use_research": true,
    "use_paper": true
}
What are the latest advances in Retrieval Augmented Generation?


2026-08-07 16:47:38,296 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:47:40,539 | INFO | Research_agent completed with 5 results
2026-08-07 16:47:40,542 | INFO | Paper_agent just started
2026-08-07 16:47:42,003 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-07 16:47:42,016 | INFO | Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=Find+recent+academic+papers+on+Retrieval+Augmented+Generation+%28RAG%29+from+2022+onwards.&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=5


...

  Faithfulness: 0.8 | Relevancy: 0.9

Evaluating 4/5: How do vector databases compare for productio...


2026-08-07 17:00:07,032 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Faithfulness: 0.8 | Relevancy: 0.7

Evaluating 5/5: What are the tradeoffs between RAG and fine-t...


2026-08-07 17:00:10,102 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Faithfulness: 0.9 | Relevancy: 0.8

   MULTI-AGENT RESEARCH ASSISTANT
   EVALUATION RESULTS
   Questions Evaluated : 5
   Faithfulness        : 0.84
   Answer Relevancy    : 0.82

 Resume bullet:
   Evaluated using LLM-as-judge across 5 test
   queries achieving faithfulness of 0.84 and
   answer relevancy of 0.82
